### Load base model (names set as *_nonlinear.pth)

In [ ]:
from ptlpinns.models import model, train_PDE
from ptlpinns.odes import numerical
import numpy as np
import torch
import time
import matplotlib.pyplot as plt

name = "model_KPP_Fisher_nonlinear.pth"
path = "/home/dda24/PTL-PINNs/ptlpinns/models/train/KPP_Fisher_nonlinear"
base_model = model.Multihead_model_PDE(k=9, bias=True)
base_model.load_state_dict(torch.load(f'{path}/{name}'))

bias = True
w_pde, w_bc, w_ic = 1, 10, 10
L, T, Nx, Nt = 2, 5, 150, 150

training_log = {
    'name': name,
    'bias': bias,
    'k' : 9,
    'domain_info':{
        'L': L,
        'T': T,},
    'Nx' : Nx,
    'Nt' : Nt,
    'w_pde': w_pde,
    'w_bc': w_bc,
    'w_ic': w_ic,}

### Initialize transfer learning model

In [ ]:
backbone = {k: v for k, v in base_model.state_dict().items() if not k.startswith("final_layers")}
transfer_model = model.Multihead_model_PDE(k=1, bias=True)
transfer_model.load_state_dict(backbone, strict=False)

for name, param in transfer_model.named_parameters():
    if not name.startswith("final_layers"):
        param.requires_grad = False

### Transfer learning parameters

In [ ]:
epsilon = 0.5
epsilons = [epsilon]
D = 0.1

def u_0_function(x):
    return (np.sin(np.pi * x / L))

def forcing_zeros(input):
    x = input[:, 0].unsqueeze(1)
    return torch.zeros_like(x)

def ic_sin(input):
    x = input[:, 0].unsqueeze(1)
    return torch.sin(torch.pi*x / L)

boundary_values = [[0,0]]
u_0 = [u_0_function for _ in range(len(epsilons))]  
forcing = [lambda x, t: 0 for _ in range(len(epsilons))] 
bcs = [[lambda t: 0, lambda t: 0] for _ in range(len(epsilons))]  
polynomial = [lambda u: -u + u**2 for _ in range(len(epsilons))]

L, T = 2, 5
x_span, t_span = (0, L), (0, T) 

Nx, Nt = 50, 50
x, t, grid = train_PDE.generate_interior_tensor(IG=(Nx, Nt), x_span=(0, L), t_span=(0, T), require_grad=False)

X_grid = np.linspace(0, L, Nx)
t_eval = np.linspace(0, T, Nt)
mesh_x, mesh_t = np.meshgrid(X_grid, t_eval)

### Define numerical solution

In [ ]:
numerical_solution = numerical.solution_KPP(epsilons, D, polynomial, x_span, t_span, Nx, Nt, u_0, forcing, bcs).squeeze().reshape(-1, 1)

### Transfer learning

In [ ]:
optimizer = torch.optim.Adam(model.head_parameters(transfer_model), lr=1e-2)
num_iter = 50000
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=300, gamma=0.95)
interior_grid = (50, 50)
Nic = Nx
Nbc = Nt
ic_weight, pde_weight, bc_weight = 1, 10, 10

In [ ]:
train_PDE.compute_transfer_learning(transfer_model=transfer_model, numerical_solution=numerical_solution, optimizer=optimizer, num_iter=num_iter,
                                    Forcing_functions=[forcing_zeros], boundary_values=boundary_values, initial_value=[ic_sin], coeff=D,
                                    interior_grid=interior_grid, x_span=x_span, t_span=t_span,
                                    Nic=Nic, Nbc=Nbc, every=100, pde_weight=pde_weight, bc_weight=bc_weight,
                                    ic_weight=ic_weight, scheduler=scheduler, epsilon=epsilon)

[iteration] 10 | total 1.191e-01 | pde 3.753e-03 | ic 2.570e-03 | bc 7.903e-03 | MAE 5.532e-02 | time 1.83
Converged at iteration 35: MAE = 0.024929218250473433 | time 6.536613516989746
